<a href="https://colab.research.google.com/github/BALAJIPASAM544/QLoRA-usecase/blob/main/Automated-Healthcare-Prior-Authorization%26Triage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Install Dependencies
In the first Colab code cell, install the necessary libraries:

In [1]:
# Install Unsloth, Hugging Face TRL, Datasets, and BitsAndBytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-wadrsn1t/unsloth_98ef367e95c240e597477f4baf1e55c3
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-wadrsn1t/unsloth_98ef367e95c240e597477f4baf1e55c3
  Resolved https://github.com/unslothai/unsloth.git to commit 4a7d860095600329be4b14005023a25a6d387010
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 36.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 54.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 68.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 36.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 55.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 61.9 MB/s  0:00:00
  Created wheel for unslot

Step 2: Load the Quantized 4-bit Foundation Model
Load Llama-3-8B-Instruct pre-quantized in 4-bit NF4 precision.

In [2]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048  # Choose max length (Llama 3 supports up to 8192)
dtype = None          # None for auto detection (FP16 on T4)
load_in_4bit = True   # Enable QLoRA 4-bit quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1112 14:39:46.203000 421 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 14:39:46.258000 421 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.4: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


Step 3: Attach Low-Rank (LoRA) Adapters
Only attach trainable low-rank matrices to attention projections and MLP layers.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # LoRA rank dimension
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,           # Scaling factor
    lora_dropout=0,          # Optimized to 0 for Unsloth kernels
    bias="none",
    use_gradient_checkpointing="unsloth",  # Saves ~30% additional VRAM
    random_state=42,
)

# Print trainable parameters to verify low-rank footprint
trainable_params, all_params = model.get_nb_trainable_parameters()
print(f"Trainable params: {trainable_params:,} / {all_params:,} ({100 * trainable_params / all_params:.2f}%)")
# Output: ~41.9M trainable params (~0.52%)

Step 4: Format Domain Data (Clinical Prior Auth Scenario)
Prepare an enterprise prompt template (Instruction, Input, Structured JSON Output).

In [ ]:
from datasets import Dataset

# Prompt template formatted for Llama-3 instruction structure
prompt_template = """Below is an unstructured clinical note. Extract the diagnosis, map it to an ICD-10 code, and provide prior authorization routing in valid JSON.

### Clinical Note:
{}

### Structured Response:
{}"""

# Realistic domain dataset examples
domain_data = [
    {
        "input": "Patient presents with persistent severe retrosternal chest pain radiating to the left jaw and shortness of breath for 45 minutes. EKG indicates acute anterior ST-segment elevation.",
        "output": '{"primary_diagnosis": "ST elevation myocardial infarction", "icd10_code": "I21.09", "department_routing": "Cardiology / Cath Lab", "prior_auth_status": "Emergency Expedited"}'
    },
    {
        "input": "Follow-up for type 2 diabetes mellitus with diabetic polyneuropathy in bilateral lower extremities. Patient reports burning sensations in feet.",
        "output": '{"primary_diagnosis": "Type 2 diabetes mellitus with diabetic polyneuropathy", "icd10_code": "E11.42", "department_routing": "Endocrinology / Neurology", "prior_auth_status": "Standard"}'
    },
    {
        "input": "Adult patient evaluated for persistent pain and stiffness in right shoulder following a lateral fall. MRI confirms complete supraspinatus tendon tear.",
        "output": '{"primary_diagnosis": "Complete rotator cuff tear of right shoulder", "icd10_code": "M75.111", "department_routing": "Orthopedic Surgery", "prior_auth_status": "Pre-Authorization Required"}'
    }
]

def format_prompts(batch):
    texts = []
    EOS_TOKEN = tokenizer.eos_token
    for inp, out in zip(batch["input"], batch["output"]):
        text = prompt_template.format(inp, out) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = Dataset.from_list(domain_data)
dataset = dataset.map(format_prompts, batched=True)

Step 5: Configure and Run SFTTrainer
Run the training loop using Hugging Face's SFTTrainer and SFTConfig.

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # Effective batch size = 8
    warmup_steps=5,
    max_steps=60,                   # Demo steps for Colab execution
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="paged_adamw_8bit",
    seed=42,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

# Execute Fine-Tuning
trainer_stats = trainer.train()

Step 6: Test Inference on Unseen Clinical Data
Run the model directly in fast inference mode to observe schema and code compliance:

In [ ]:
FastLanguageModel.for_inference(model)

test_note = "Patient sustained traumatic twisting injury to right knee while playing soccer. Joint effusion present; pivot-shift test positive, suspecting anterior cruciate ligament rupture."

inputs = tokenizer(
    [prompt_template.format(test_note, "")],
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.1
)

decoded_response = tokenizer.batch_decode(outputs)
print(decoded_response[0].split("### Structured Response:\n")[1])

Step 7: Export and Save the Model
Save the lightweight LoRA adapters (files are under ~170 MB):

In [ ]:
# Save LoRA adapter weights locally in Colab
model.save_pretrained("qlora_clinical_adapter")
tokenizer.save_pretrained("qlora_clinical_adapter")

# Or merge weights for standalone 16-bit deployment
# model.save_pretrained_merged("final_merged_model", tokenizer, save_method="merged_16bit")